In [ ]:
import multiprocessing
import os

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # Use if using compilation-dependent reproducibility

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from piano import Composer, time_code

try:
    import rapids_singlecell as rsc
    use_rapids = True
    print('Using rapids singlecell to speed up postprocessing', flush=True)
except:
    print('Warning: Unable to use rapids singlecell in this environment', flush=True)
    use_rapids = False
np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)

In [ ]:
# Run parameters
run_name = 'piano_hmba_bg'
outdir = f'results/{run_name}'  # Relative to the notebook's folder
os.makedirs(f'{outdir}/integration_results', exist_ok=True)
os.makedirs(f'{outdir}/figures', exist_ok=True)

print(f'Number of CPU cores: {multiprocessing.cpu_count()}', flush=True)
print(f'Number of GPUs: {torch.cuda.device_count()}', flush=True)
print(f'CUDA GPUs available: {torch.cuda.is_available()}', flush=True)

## Example data: HMBA basal ganglia

This tutorial integrates human, macaque and marmoset basal ganglia single-nucleus RNA-seq from the Human and Mammalian Brain Atlas (HMBA), restricted to genes with one-to-one orthologs across the three species. Prepare this file yourself from the HMBA basal ganglia data (map genes to one-to-one orthologs, keep raw counts) and save it as `data/hmba_bg.h5ad`: an AnnData with raw counts in `X` and these `obs` columns:

- `organism` (species) and `donor_id`: the batch effects to correct
- `Neighborhood`, `Class`, `Subclass` and `Group`: cell-type levels of the HMBA taxonomy, coarse to fine, used to colour the UMAPs


In [ ]:
with time_code('Load Anndata'):
    adata = sc.read_h5ad('data/hmba_bg.h5ad')
    print(f"Training on: {adata}")

In [ ]:
with time_code('Initialize Pianist'):
    pianist = Composer(
        adata,
        categorical_covariate_keys = ['organism', 'donor_id'],             # Discrete batch effects: species and donor
        continuous_covariate_keys = [],                                    # Continuous sources of variation
        n_top_genes=4096,                                                  # Number of highly variable genes to use
        hvg_batch_key='organism',                                          # Used for SeuratV3 highly variable gene (HVG) selection
        memory_mode='GPU',   # Use GPU
        compile_model=True,  # On a GPU or the CPU; the CPU needs a C++ compiler (e.g. g++)
        num_workers=0,       # Must be 0 for GPU and SparseGPU memory modes, otherwise ~11 workers
        deterministic=True,  # Requires CUBLAS_WORKSPACE_CONFIG=:4096:8 if compiling
        # Other settings keep the 0.2.0 defaults; the README says when to change them (e.g. a stronger adversary when
        # batches stay apart)
        run_name=run_name,
        outdir=outdir,
    )

In [ ]:
with time_code('Run pipeline'):
    pianist.run_pipeline()

In [ ]:
with time_code('Save pianist'):
    pianist.save(f'{outdir}/pianist.pkl' )

In [ ]:
# Use if loading a previously saved model. Optional if pianist is already initialized/trained
with time_code('Load model'):
    pianist = Composer.load(f'{outdir}/pianist.pkl')
    pianist.load_model(f'{outdir}/checkpoints/model_checkpoint.pt')

In [ ]:
with time_code('Get Latent Representation'):
    adata.obsm['X_PIANO'] = pianist.get_latent_representation(adata)  # adata parameter is optional if not loading saved model from disk

In [ ]:
with time_code('Create Latent Representation DataFrame'):
    X_PIANO_df = pd.DataFrame(adata.obsm['X_PIANO'], index=adata.obs_names)

In [ ]:
with time_code('Save Latent Space CSV'):
    X_PIANO_df.to_csv(f'{outdir}/integration_results/X_PIANO.csv')

In [ ]:
with time_code('Computing Neighbors'):
    if use_rapids:  # GPU acceleration
        rsc.pp.neighbors(
            adata, n_neighbors=15,
            n_pcs=pianist.model.latent_size,
            use_rep='X_PIANO',
            random_state=0,
        )
    else:
        sc.pp.neighbors(
            adata, n_neighbors=15,
            n_pcs=pianist.model.latent_size,
            use_rep='X_PIANO',
            random_state=0,
        )

In [ ]:
with time_code('Computing UMAP'):
    if use_rapids:  # GPU acceleration
        rsc.tl.umap(adata, random_state=0)
    else:
        sc.tl.umap(adata, random_state=0)

In [ ]:
with time_code('Creating UMAP DataFrame'):
    UMAP_df = pd.DataFrame(
        adata.obsm['X_umap'],
        index=adata.obs_names,
    )

In [ ]:
with time_code('Saving UMAP CSV'):
    UMAP_df.to_csv(f'{outdir}/integration_results/UMAP_2D_df.csv')

In [ ]:
with time_code('Possibly saving Anndata'):
    adata.write_h5ad(f'{outdir}/integration_results/adata_integrated.h5ad')

In [ ]:
%matplotlib inline
with time_code('Plotting UMAPs'):
    print(adata, flush=True)
    for umap_label in ['organism', 'donor_id', 'Neighborhood', 'Class', 'Subclass', 'Group']:
        fig = sc.pl.umap(
            adata[np.random.permutation(np.arange(adata.shape[0]))],
            color=umap_label,
            return_fig=True,
        )
        legend = fig.axes[0].get_legend()
        if legend is not None:
            legend.set_bbox_to_anchor((0.5, -0.1))  # Adjust legend coordinates
            legend.set_loc('upper center')          # Actually positions legend below
        fig.savefig(
            f'{outdir}/figures/UMAP_{umap_label}.png', bbox_inches='tight'
        )
        plt.show()
        plt.close(fig)